In [ ]:
# Kaggriculture RL -- random-profile LEAGUE games (generated by python/league_kaggle.py; do not hand-edit the pushed copy).
# Runs the static Rust `league-rand` (dataset debmalya84/kaggriculture-rl-bin) for batches FIRST .. FIRST+BATCHES-1
# and writes /kaggle/working/league/batch-*.{tsv,obs} + manifest.txt + run.json (finished_utc, so a stale
# output is never filed). Batch k always covers seeds 10,000,000 + k*500, so outputs of parallel
# notebooks never collide.
import datetime, glob, json, os, shutil, subprocess, time
T0 = time.time()
FIRST = 74
BATCHES = 23
THREADS = 4
W = '/kaggle/working/w'
os.makedirs(W + '/base', exist_ok=True)
src = os.path.dirname(sorted(glob.glob('/kaggle/input/**/league-rand', recursive=True))[0])
shutil.copy(src + '/league-rand', W + '/league-rand'); os.chmod(W + '/league-rand', 0o755)
shutil.copy(src + '/v611_router.json', W + '/base/router.json')
shutil.copy(src + '/v611_routes.json', W + '/base/routes.json')
shutil.copy(src + '/profiles_v2.json', W + '/profiles.json')
sha = open(src + '/SHA256-league.txt').read().split()[0]
print('binary', src, sha, 'cpus', os.cpu_count(), flush=True)
OUT = '/kaggle/working/league'
cmd = [W + '/league-rand', '--base', W + '/base', '--profiles', W + '/profiles.json', '--out', OUT,
       '--first', str(FIRST), '--batches', str(BATCHES), '--batch', '500', '--threads', str(THREADS)]
print(' '.join(cmd), flush=True)
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end='', flush=True)
rc = p.wait()
files = sorted(os.path.relpath(f, '/kaggle/working') for f in glob.glob(OUT + '/batch-*') if not f.endswith('.part'))
with open(OUT + '/manifest.txt', 'w') as fh:
    fh.write('\n'.join(files + ['league/manifest.txt']) + '\n')
json.dump({'rc': rc, 'secs': round(time.time() - T0, 1), 'files': len(files) + 1, 'first': FIRST, 'batches': BATCHES,
           'binary_sha': sha, 'finished_utc': datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')},
          open('/kaggle/working/run.json', 'w'), indent=1)
shutil.rmtree(W, ignore_errors=True)
print('RC', rc, 'files', len(files), 'secs', round(time.time() - T0, 1), flush=True)
assert rc == 0, 'league-rand failed'
